# 候选因子01：库存周期
BigAlpha 2026 因子提交文件。核心入口为 `main(datasources, start_date, end_date)`。

In [ ]:
from __future__ import annotations


def main(datasources, start_date, end_date):
    """Inventory rerating with recent sell-through gated restocking.

    Economic hypothesis: once inventory growth is aligned with sales and cash
    conversion, stocks that have recently underperformed can be rerated as the
    market recognizes restocking is active rather than involuntary. Inventory
    growth deserves a stronger reward only when recent inventory-to-sales is
    also improving, because that means new stock is being sold through rather
    than merely added to the balance sheet. Recent deterioration converts the
    same inventory build into fresh overhang risk.
    """
    import numpy as np
    import pandas as pd
    import dai

    start = pd.Timestamp(start_date).normalize()
    end = pd.Timestamp(end_date).normalize()
    fin_start = start - pd.Timedelta(days=540)
    bar_start = start - pd.Timedelta(days=100)

    def _clean_keys(frame):
        frame = frame.copy()
        frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
        frame["instrument"] = frame["instrument"].astype(str)
        return frame

    def _safe_divide(numerator, denominator):
        denom = denominator.abs().where(denominator.abs() > 1.0e-12)
        return (numerator / denom).replace([np.inf, -np.inf], np.nan)

    def _center_rank(series):
        dates = series.index.get_level_values("date")
        ranked = series.groupby(dates, sort=False).rank(pct=True, method="average") - 0.5
        return ranked - ranked.groupby(dates, sort=False).transform("mean")

    def _rank_frame(frame):
        return frame.apply(_center_rank)

    financial = dai.query(
        f"""
        SELECT
            date,
            instrument,
            category,
            shift,
            inventories,
            total_current_assets,
            total_assets,
            operating_revenue,
            operating_costs,
            operating_revenue - operating_costs AS gross_profit,
            net_cffoa,
            cash_received_from_sales_and_services,
            cash_paid_for_goods_and_services
        FROM {datasources['financial']}
        WHERE shift = 0
          AND category IN ('lf', 'ttm')
        """,
        filters={"date": [fin_start, end]},
    ).df()

    fin_signal = pd.DataFrame(columns=["date", "instrument", "alignment_signal"])
    if not financial.empty:
        financial = _clean_keys(financial)
        financial["category"] = financial["category"].astype(str)
        numeric_cols = [
            "inventories",
            "total_current_assets",
            "total_assets",
            "operating_revenue",
            "operating_costs",
            "gross_profit",
            "net_cffoa",
            "cash_received_from_sales_and_services",
            "cash_paid_for_goods_and_services",
        ]
        for col in numeric_cols:
            financial[col] = pd.to_numeric(financial[col], errors="coerce")
        financial = financial.sort_values(["instrument", "date", "category"]).drop_duplicates(
            ["date", "instrument", "category"], keep="last"
        )
        lf = financial[financial["category"] == "lf"][
            ["date", "instrument", "inventories", "total_current_assets", "total_assets"]
        ]
        ttm = financial[financial["category"] == "ttm"][
            [
                "date",
                "instrument",
                "operating_revenue",
                "operating_costs",
                "gross_profit",
                "net_cffoa",
                "cash_received_from_sales_and_services",
                "cash_paid_for_goods_and_services",
            ]
        ]
        features = lf.merge(ttm, on=["date", "instrument"], how="outer").sort_values(
            ["instrument", "date"]
        )
        grouped = features.groupby("instrument", sort=False)
        inventory = features["inventories"].clip(lower=0.0)
        prior_inventory = grouped["inventories"].shift(126).clip(lower=0.0)
        recent_inventory = grouped["inventories"].shift(63).clip(lower=0.0)
        revenue = features["operating_revenue"]
        prior_revenue = grouped["operating_revenue"].shift(126)
        recent_revenue = grouped["operating_revenue"].shift(63)
        inv_growth = np.log1p(inventory) - np.log1p(prior_inventory)
        revenue_growth = _safe_divide(revenue - prior_revenue, prior_revenue.abs() + 1.0)
        growth_alignment = revenue_growth - inv_growth

        inventory_to_sales = _safe_divide(inventory, revenue.abs() + 1.0)
        prior_inventory_to_sales = _safe_divide(prior_inventory, prior_revenue.abs() + 1.0)
        inventory_sales_normalization = prior_inventory_to_sales - inventory_to_sales
        recent_inventory_to_sales = _safe_divide(recent_inventory, recent_revenue.abs() + 1.0)
        recent_inventory_sales_normalization = recent_inventory_to_sales - inventory_to_sales

        turnover = _safe_divide(features["operating_costs"], inventory)
        turnover_prior = _safe_divide(grouped["operating_costs"].shift(126), prior_inventory)
        turnover_change = turnover - turnover_prior
        margin = _safe_divide(features["gross_profit"], revenue)
        margin_prior = _safe_divide(grouped["gross_profit"].shift(126), prior_revenue)
        margin_change = margin - margin_prior
        cash_margin = _safe_divide(features["net_cffoa"], revenue)
        cash_prior = _safe_divide(grouped["net_cffoa"].shift(126), prior_revenue)
        cash_change = cash_margin - cash_prior
        cash_conversion = _safe_divide(
            features["cash_received_from_sales_and_services"]
            - features["cash_paid_for_goods_and_services"],
            revenue,
        )
        inventory_intensity = _safe_divide(inventory, features["total_current_assets"])

        idx = pd.MultiIndex.from_frame(features[["date", "instrument"]])
        parts = pd.DataFrame(
            {
                "growth_alignment": growth_alignment.to_numpy(),
                "inventory_sales_normalization": inventory_sales_normalization.to_numpy(),
                "recent_inventory_sales_normalization": recent_inventory_sales_normalization.to_numpy(),
                "turnover_change": turnover_change.to_numpy(),
                "margin_change": margin_change.to_numpy(),
                "cash_change": cash_change.to_numpy(),
                "cash_conversion": cash_conversion.to_numpy(),
                "inventory_intensity": inventory_intensity.to_numpy(),
                "inv_growth": inv_growth.to_numpy(),
            },
            index=idx,
        )
        ranked = _rank_frame(parts)
        demand_match = (
            0.32 * ranked["growth_alignment"]
            + 0.18 * ranked["inventory_sales_normalization"]
            + 0.08 * ranked["recent_inventory_sales_normalization"]
            + 0.22 * ranked["turnover_change"]
            + 0.10 * ranked["margin_change"]
            + 0.10 * ranked["cash_change"]
        )
        cash_support = 0.65 * ranked["cash_conversion"] + 0.35 * ranked["cash_change"]
        excess_inventory = ranked["inv_growth"].clip(lower=0.0) * (-demand_match).clip(
            lower=0.0
        )
        balanced_restocking = ranked["inv_growth"].clip(lower=0.0) * demand_match.clip(
            lower=0.0
        )
        sellthrough_restocking = balanced_restocking * ranked[
            "recent_inventory_sales_normalization"
        ].clip(lower=0.0)
        recent_overhang = ranked["inv_growth"].clip(lower=0.0) * (
            -ranked["recent_inventory_sales_normalization"]
        ).clip(lower=0.0)
        score = 0.58 * demand_match + 0.24 * cash_support
        score = score + 0.36 * balanced_restocking + 0.16 * sellthrough_restocking
        score = score - 0.55 * excess_inventory - 0.08 * recent_overhang
        score = score - 0.12 * ranked["inventory_intensity"]
        fin_signal = features[["date", "instrument"]].copy()
        fin_signal["alignment_signal"] = score.to_numpy()

    bars = dai.query(
        f"""
        SELECT
            CAST(date_trunc('day', date) AS TIMESTAMP) AS date,
            instrument,
            arg_min(open, date) AS open,
            arg_max(close, date) AS close,
            SUM(volume) AS volume,
            SUM(amount) AS amount
        FROM {datasources['bar1m']}
        GROUP BY 1, 2
        ORDER BY 2, 1
        """,
        filters={"date": [bar_start, end]},
    ).df()

    market = pd.DataFrame(columns=["date", "instrument", "market_signal"])
    if not bars.empty:
        bars = _clean_keys(bars)
        for col in ["open", "close", "volume", "amount"]:
            bars[col] = pd.to_numeric(bars[col], errors="coerce")
        bars = bars.sort_values(["instrument", "date"]).drop_duplicates(
            ["date", "instrument"], keep="last"
        )
        grouped_bars = bars.groupby("instrument", sort=False)
        ret_1 = _safe_divide(bars["close"], grouped_bars["close"].shift(1)) - 1.0
        bars["_up_amount"] = bars["amount"].where(ret_1 > 0.0, 0.0)
        bars["_down_amount"] = bars["amount"].where(ret_1 <= 0.0, 0.0)
        bars["_up_day"] = (ret_1 > 0.0).astype(float)
        grouped_bars = bars.groupby("instrument", sort=False)
        up_amount = grouped_bars["_up_amount"].transform(
            lambda x: x.rolling(20, min_periods=5).sum()
        )
        down_amount = grouped_bars["_down_amount"].transform(
            lambda x: x.rolling(20, min_periods=5).sum()
        )
        down_amount_ratio = _safe_divide(down_amount, up_amount + down_amount)
        fast_amount = grouped_bars["amount"].transform(
            lambda x: x.rolling(10, min_periods=4).mean()
        )
        slow_amount_20 = grouped_bars["amount"].transform(
            lambda x: x.rolling(20, min_periods=6).mean()
        )
        slow_amount_40 = grouped_bars["amount"].transform(
            lambda x: x.rolling(40, min_periods=10).mean()
        )
        amount_cooldown = -(
            _safe_divide(fast_amount, slow_amount_40.combine_first(slow_amount_20)) - 1.0
        )
        ret_20 = _safe_divide(bars["close"], grouped_bars["close"].shift(20)) - 1.0
        ret_40 = _safe_divide(bars["close"], grouped_bars["close"].shift(40)) - 1.0
        ret_60 = _safe_divide(bars["close"], grouped_bars["close"].shift(60)) - 1.0
        ret_long = ret_60.combine_first(ret_40).combine_first(ret_20)
        reversal = -(_safe_divide(bars["close"], bars["open"]) - 1.0)
        recent_stabilization = _safe_divide(bars["close"], grouped_bars["close"].shift(3)) - 1.0
        contrarian_base = (-ret_long).clip(lower=0.0)
        reversal_confirmation = contrarian_base * recent_stabilization.clip(lower=0.0)
        falling_knife = contrarian_base * (-recent_stabilization).clip(lower=0.0)
        idx = pd.MultiIndex.from_frame(bars[["date", "instrument"]])
        market_parts = pd.DataFrame(
            {
                "contrarian_20": (-ret_20).to_numpy(),
                "contrarian_long": (-ret_long).to_numpy(),
                "down_amount_ratio": down_amount_ratio.to_numpy(),
                "amount_cooldown": amount_cooldown.to_numpy(),
                "stabilization": recent_stabilization.to_numpy(),
                "reversal_confirmation": reversal_confirmation.to_numpy(),
                "falling_knife": falling_knife.to_numpy(),
                "intraday_reversal": reversal.to_numpy(),
            },
            index=idx,
        )
        ranked_market = _rank_frame(market_parts).fillna(0.0)
        bars["market_signal"] = (
            0.18 * ranked_market["contrarian_20"]
            + 0.12 * ranked_market["contrarian_long"]
            + 0.14 * ranked_market["down_amount_ratio"]
            + 0.10 * ranked_market["amount_cooldown"]
            + 0.08 * ranked_market["stabilization"]
            + 0.22 * ranked_market["reversal_confirmation"]
            - 0.18 * ranked_market["falling_knife"]
            + 0.06 * ranked_market["intraday_reversal"]
        ).to_numpy()
        market = bars[["date", "instrument", "market_signal"]]

    if "instruments" in datasources:
        universe = dai.query(
            f"SELECT date, instrument FROM {datasources['instruments']}",
            filters={"date": [start, end]},
        ).df()
        universe = _clean_keys(universe)
    else:
        # 官方只替换 bar1m / financial；股票池是固定辅助表。必须以官方
        # instruments 为输出网格，避免财务自然日和池外股票混入提交结果。
        universe = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={"date": [start, end]},
        ).df()
        universe = _clean_keys(universe)

    output_dates = pd.DataFrame({"date": sorted(universe["date"].dropna().unique())})
    # 隐藏/自检行情表可能在评估尾日没有当日分钟明细。此时沿用最近一个
    # 已发生交易日的市场信号，避免把全截面回填为0并在官方标准化后整日变NaN。
    if not market.empty and not output_dates.empty:
        daily_market = []
        for instrument, group in market.sort_values(["instrument", "date"]).groupby(
            "instrument", sort=False
        ):
            filled = output_dates.merge(group, on="date", how="left").sort_values("date")
            filled["instrument"] = instrument
            filled["market_signal"] = filled["market_signal"].ffill()
            daily_market.append(filled)
        if daily_market:
            market = pd.concat(daily_market, ignore_index=True)

    scored = universe.copy()
    if not fin_signal.empty and not output_dates.empty:
        daily_fin = []
        for instrument, group in fin_signal.sort_values(["instrument", "date"]).groupby(
            "instrument", sort=False
        ):
            filled = output_dates.merge(group, on="date", how="left").sort_values("date")
            filled["instrument"] = instrument
            filled["alignment_signal"] = filled["alignment_signal"].ffill()
            daily_fin.append(filled)
        if daily_fin:
            daily_financial = pd.concat(daily_fin, ignore_index=True)
            scored = scored.merge(daily_financial, on=["date", "instrument"], how="left")
        else:
            scored["alignment_signal"] = np.nan
    else:
        scored["alignment_signal"] = np.nan

    scored = scored.merge(market, on=["date", "instrument"], how="left")
    scored["alignment_signal"] = scored["alignment_signal"].fillna(0.0)
    scored["market_signal"] = scored["market_signal"].fillna(0.0)
    scored["raw"] = 0.58 * scored["alignment_signal"] + 0.42 * scored["market_signal"]
    ranked = scored.groupby("date", sort=False)["raw"].rank(pct=True, method="average") - 0.5
    scored["factor"] = ranked - ranked.groupby(scored["date"], sort=False).transform("mean")

    result = scored[["date", "instrument", "factor"]].copy()
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    result = result.loc[result["date"].between(start, end)].drop_duplicates(
        ["date", "instrument"], keep="last"
    )
    return result.sort_values(["date", "instrument"]).reset_index(drop=True)
